In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

DB="customer_health"
BRONZE="BRONZE"
SILVER="SILVER"

spark.conf.set("spark.sql.session.timeZone","UTC")

PARSED_TABLE=f"{DB}.{SILVER}.LEAD_ACTIVITIES_PARSED"
EMAIL_TABLE=f"{DB}.{SILVER}.LEAD_ACTIVITIES_EMAIL"

activity_schema="""struct<_type:string,lead_id:string,contact_id:string,user_id:string,id:string,
activity_at:string,date_created:string,date_updated:string,text:string,body_text:string,note:string,
old_status_label:string,new_status_label:string,created_by_name:string,updated_by_name:string,
attendees:string,custom_activity_type_id:string,starts_at:string,ends_at:string,status:string,
created_by:string,updated_by:string,attachments:string,envelope:string,sender:string,opens:string,
direction:string,subject:string,user_name:string,template_id:string,template_name:string,
date_sent:string,has_reply:string>"""

try:
    raw_df=spark.table(f"{DB}.{BRONZE}.LEAD_ACTIVITES_RAW").cache()
    raw_df.count()

    parsed_df=(raw_df
        .withColumn("activity_json_str",
            F.expr("customer_health.silver.parse_activity_final(raw_data)"))
        .withColumn("activity_array",
            F.from_json("activity_json_str","array<string>"))
        .withColumn("act_str",F.explode("activity_array"))
        .withColumn("act",F.from_json("act_str",activity_schema))
        .select(
            "insert_date",
            F.col("act.id").alias("activity_id"),
            F.col("act._type").alias("activity_type"),
            F.col("act.lead_id").alias("lead_id"),
            F.col("act.contact_id").alias("contact_id"),
            F.col("act.user_id").alias("user_id"),
            F.col("act.activity_at").cast("timestamp").alias("activity_at"),
            F.col("act.date_created").cast("timestamp").alias("date_created"),
            F.col("act.date_updated").cast("timestamp").alias("date_updated"),
            F.col("act.text").alias("text"),
            F.col("act.body_text").alias("body_text"),
            F.col("act.note").alias("note"),
            F.col("act.old_status_label").alias("old_status_label"),
            F.col("act.new_status_label").alias("new_status_label"),
            F.col("act.created_by_name").alias("created_by_name"),
            F.col("act.updated_by_name").alias("updated_by_name"),
            F.col("act.attendees").alias("attendees"),
            F.col("act.custom_activity_type_id").alias("custom_activity_type_id"),
            F.col("act.starts_at").cast("timestamp").alias("starts_at"),
            F.col("act.ends_at").cast("timestamp").alias("ends_at"),
            F.col("act.status").alias("status"),
            F.col("act.created_by").alias("created_by"),
            F.col("act.updated_by").alias("updated_by"),
            F.col("act.attachments").alias("attachments"),
            F.col("act.envelope").alias("envelope"),
            F.col("act.sender").alias("sender"),
            F.col("act.opens").alias("opens"),
            F.col("act.direction").alias("direction"),
            F.col("act.subject").alias("subject"),
            F.col("act.user_name").alias("user_name"),
            F.col("act.template_id").alias("template_id"),
            F.col("act.template_name").alias("template_name"),
            F.col("act.date_sent").cast("timestamp").alias("date_sent"),
            F.col("act.has_reply").alias("has_reply")
        ).cache())

    parsed_df.count()

    parsed_df.write.format("delta").mode("overwrite")\
        .option("overwriteSchema","true").saveAsTable(PARSED_TABLE)

    email_df=(parsed_df.filter(F.col("activity_type")=="Email")
        .select(
            "lead_id","activity_at",
            F.col("attachments").alias("attachments_json"),
            "body_text","date_created","date_updated","direction",
            F.col("envelope").alias("envelope_json"),
            F.col("sender").alias("sender_json"),
            "activity_id",
            F.col("opens").alias("opens_json"),
            "sender","status","subject","user_id","user_name",
            "template_id","template_name","date_sent","has_reply","insert_date"
        ))

    w=Window.partitionBy("activity_id","activity_at","date_sent")\
        .orderBy(F.col("insert_date").desc())

    email_df=email_df.withColumn("rn",F.row_number().over(w))\
        .filter("rn=1").drop("rn")

    if not spark.catalog.tableExists(EMAIL_TABLE):
        email_df.write.format("delta").saveAsTable(EMAIL_TABLE)
    else:
        tgt=DeltaTable.forName(spark,EMAIL_TABLE)

        update_map={c:f"src.{c}" for c in [
            "lead_id","attachments_json","body_text","date_created",
            "date_updated","direction","envelope_json","sender_json",
            "opens_json","sender","status","subject","user_id",
            "user_name","template_id","template_name","has_reply"]}
        update_map["insert_date"]="current_timestamp()"

        insert_map={c:f"src.{c}" for c in email_df.columns}
        insert_map["insert_date"]="current_timestamp()"

        (tgt.alias("tgt")
         .merge(email_df.alias("src"),
                """tgt.activity_id=src.activity_id
                   and tgt.activity_at=src.activity_at
                   and tgt.date_sent=src.date_sent""")
         .whenMatchedUpdate(set=update_map)
         .whenNotMatchedInsert(values=insert_map)
         .execute())

    raw_df.unpersist()
    parsed_df.unpersist()

    print("Notebook 1 completed successfully")

except Exception as e:
    raise Exception(f"Notebook 1 failed: {e}")